# Finding flux vacua

**What's in this notebook?** This notebook demonstrates how to use numerical optimisers to construct flux vacua by either
finding solutions to the $F$-term conditions $D_IW=0$ or to the extremum conditions $\partial_I V=0$ (with $D_IW\neq 0$).

(*Created:* Andreas Schachner, June 25, 2024)

## Imports

### General imports

In [1]:
import sys, os, warnings, time
import numpy as np
from tqdm.auto import tqdm
from functools import partial
from typing import Any, Callable, Sequence
from IPython.display import clear_output

warnings.filterwarnings('ignore')

### JAX imports

In [2]:
from jax import jit, vmap
import jax 
import jax.numpy as jnp
jax.config.update("jax_enable_x64", True)

### Plotting tools

In [3]:
import seaborn as sn
import matplotlib.pyplot as plt
import matplotlib as mpl
cmap=sn.color_palette("viridis", as_cmap=True)

### Custom library for EFT

In [4]:
import jaxvacua

TODO: Add function that converts M,K fluxes to full fluxes with and without coni-LCS?


## Finding flux vacua at $h^{1,2}=2$

We look at the typical example of the degree 18 hypersurface in $\mathbb{CP}[1,1,1,6,9]$

In [5]:
h12 = 2
model = jaxvacua.flux_sector(h12=h12,model_ID=1,model_type="KS",maximum_degree=2)

### Solutions to the $F$-term conditions

First, let us look at the following solution

In [6]:
# Moduli values
z0 = jnp.array([0.31686516+3.2301486j , 0.37113099+3.07462404j])

# Axio-dilaton values
tau0 = -0.29834961+6.80357615j

# Choices of fluxes
fluxes = jnp.array([4.,  -3.,  -2.,  -2.,   3.,   2.,  39., -13.,  -4.,   0.,  -0.,  0.])

This is a solution to the F-term conditions $D_IW=0$ as can be verified by computing

In [7]:
model.DW(z0,jnp.conj(z0),tau0,jnp.conj(tau0),fluxes)

Array([ 1.76079794e-07+4.62797516e-08j,  5.76977897e-08+1.62216480e-08j,
       -3.64220440e-08+1.51143098e-08j], dtype=complex128)

Of course, the numerical tolerance $|D_IW|<\epsilon$ is only as good as the precision used above. 

We now select some starting point and try to find the above solution through numerical minimisation.

In [8]:
# Set starting point
moduli = 2*jnp.ones(h12)*1j
tau = 1j*2.

First, let us try **Newton's method** 

In [9]:
# Set parameters for minimsation
step_size_Newton = 1
tol = 1e-10

print("Jit compiling...")

dDW = model.dDW_real(moduli,tau,fluxes)
DW = model.DW(moduli,jnp.conj(moduli),tau,jnp.conj(tau),fluxes)

print("Start minimisation...")
counter = 0
tic = time.time()
res = 10.
while res>tol:

    dDW = model.dDW_real(moduli,tau,fluxes)
    DW = model.DW(moduli,jnp.conj(moduli),tau,jnp.conj(tau),fluxes)
    DW_real = jnp.concatenate([jnp.real(DW),jnp.imag(DW)])
    deltaPhi = -jnp.linalg.inv(dDW)@DW_real
    
    moduli = moduli + step_size_Newton*(deltaPhi[h12+1:2*h12+1]+1j*deltaPhi[0:h12])
    tau = tau + step_size_Newton*(deltaPhi[2*h12+1]+1j*deltaPhi[h12])
    
    res = jnp.sum(jnp.abs(DW_real))
    counter+=1
    
    print(f"residual: {res}                     counter: {counter}           ", end="\r")
    
    if counter>10**3:
        print("")
        print(f"STOPPED RUN: residual: {res}                     counter: {counter}           ")
        break
        
toc = time.time()
print("")
print(f"Finished in {np.around((toc-tic)*1000,2)}ms.")
print(f"Difference to z0: {jnp.linalg.norm(z0-moduli)}")
print(f"Difference to tau0: {jnp.abs(tau0-tau)}")

Jit compiling...
Start minimisation...
residual: 9.947598300641403e-14                     counter: 7           
Finished in 1676.48ms.
Difference to z0: 5.240817230941895e-09
Difference to tau0: 4.529881897792484e-09


After `jit` compilation, the minimsation is very fast (of order a few milliseconds).

We can also employ more sophisticated minimsation algorithms e.g. from the optimisation library `scipy.optimize.`.
Here, we can use [`scipy.optimize.root`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.root.html) as follows

In [10]:
from scipy.optimize import root

eps=1e-10
x0=np.array([0,1,0,1,0,1.])*2

result = root(model.DW_x,x0,args=(fluxes,), jac=model.dDW_x, method='hybr',tol=eps)

result

 message: The solution converged.
 success: True
  status: 1
     fun: [ 4.007e-12 -1.728e-11  1.478e-12 -5.130e-12 -1.342e-11
           -6.132e-12]
       x: [ 3.169e-01  3.230e+00  3.711e-01  3.075e+00 -2.983e-01
            6.804e+00]
  method: hybr
    nfev: 23
    njev: 1
    fjac: [[-7.867e-01  2.146e-01 ... -3.093e-01 -4.211e-01]
           [-4.887e-01 -1.161e-02 ...  7.219e-01  4.524e-01]
           ...
           [-2.443e-01 -9.283e-01 ... -2.279e-01  1.200e-01]
           [ 2.393e-01 -7.081e-02 ... -1.141e-01  1.611e-01]]
       r: [ 4.204e+01  5.225e+01 ...  9.987e+00  3.999e-01]
     qtf: [ 8.237e-12  1.240e-10 -6.844e-11 -1.674e-11 -1.542e-10
            4.539e-12]

Let us check how close the solution is to the expected one, we compute

In [11]:
X = result.x
X = X[::2]+1j*X[1::2]

print(f"Difference to z0: {jnp.linalg.norm(z0-X[:2])}")
print(f"Difference to tau0: {jnp.abs(tau0-X[2])}")

Difference to z0: 5.239981291303855e-09
Difference to tau0: 4.531281031645634e-09


We can now verify explicitly that the above solution corresponds to a minimum of the potential by looking at the Hessian:

In [12]:
z,cz,t,ct = model._convert_real_to_complex(result.x)
H = model.hessian(z,cz,t,ct,fluxes)
H_evals = jnp.linalg.eigvalsh(H)
H_evals

Array([1.41619802e-02, 7.04807187e-02, 3.11152081e-01, 4.80845026e-01,
       3.38536582e+00, 1.72933512e+01], dtype=float64)

For a SUSY minimum, it can sometimes be convenient to use the faster implementation using `mode="SUSY"`:

In [13]:
H_SUSY = model.hessian(z,cz,t,ct,fluxes,mode="SUSY")
H_SUSY_evals = jnp.linalg.eigvalsh(H_SUSY)
H_evals/H_SUSY_evals

Array([1., 1., 1., 1., 1., 1.], dtype=float64)

We see that the eigenvalues are almost exactly the same. 
**CAVEAT:** Having said that, numerically setting $D_I W=0$ in the code might lead to numerical errors that may become dangerous
in siutations where high precision is required.

We can compare the time difference

In [14]:
%%timeit
H = jax.block_until_ready(model.hessian(z,cz,t,ct,fluxes))

422 μs ± 20.1 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [15]:
%%timeit
H_SUSY = jax.block_until_ready(model.hessian(z,cz,t,ct,fluxes,mode="SUSY"))

76.9 μs ± 1.29 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


We can compue the masses of the fields as follows

In [16]:
M = model.mass_matrix(z,cz,t,ct,fluxes)
masses = jnp.linalg.eigvalsh(M)
masses

Array([  7.04573637,  10.30979665,  40.57054696, 151.77961456,
       165.8015113 , 479.62206571], dtype=float64)

We can again speed up the computation by using `mode="SUSY"`

In [17]:
M_SUSY = model.mass_matrix(z,cz,t,ct,fluxes,mode="SUSY")
masses_SUSY = jnp.linalg.eigvalsh(M_SUSY)
masses_SUSY/masses

Array([1., 1., 1., 1., 1., 1.], dtype=float64)

This gives again the same output. We again find a significant speed-up:

In [18]:
%%timeit
M = jax.block_until_ready(model.mass_matrix(z,cz,t,ct,fluxes))

561 μs ± 10.5 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [19]:
%%timeit
M_SUSY = jax.block_until_ready(model.mass_matrix(z,cz,t,ct,fluxes,mode="SUSY"))

171 μs ± 792 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


### Non-supersymmetric minima with $D_IW\neq 0$

First, let us look at the following solution

In [20]:
# Moduli values
z0 = jnp.array([1.19 + 1.01j , 2.94 + 3.86j])

# Axio-dilaton values
tau0 = -0.40 + 5.30j

# Choices of fluxes
fluxes = jnp.array([0., 18., 3., 2., 4., -4., 0., -7., 0., 0., 0., 1.])

This solution was presented in appendix A of [2308.15525](https://arxiv.org/pdf/2308.15525) (solution `c` in Tables 2 and 3).
It corresponds to a dS minimum of the potential

In [21]:
model.dV(z0,jnp.conj(z0),tau0,jnp.conj(tau0),fluxes)

Array([-0.00609127+0.03064825j, -0.00062935+0.00231964j,
       -0.00099841-0.0007843j ], dtype=complex128)

at which the $F$-terms are non-vanishing $D_IW\neq 0$ as can be verified by computing

In [22]:
model.DW(z0,jnp.conj(z0),tau0,jnp.conj(tau0),fluxes)

Array([-0.09850933-0.87411864j, -0.03901691-0.02279752j,
        0.13960455-0.03094916j], dtype=complex128)

Of course, the numerical tolerance $|\partial_I V|<\epsilon$ is only as good as the precision used above which we improve with the
numerical optimiser below.

We now select some starting point and try to find the above solution through numerical minimisation.

In [23]:
# Set starting point
moduli = 1+jnp.ones(h12)*1j
tau = 1j*1.

First, let us try **Newton's method** 

In [24]:
# Set parameters for minimsation
step_size_Newton = 0.1
tol = 1e-12

print("Jit compiling...")

x = model._convert_complex_to_real(moduli,jnp.conj(moduli),tau,jnp.conj(tau))

#ddV = model.ddV(moduli,jnp.conj(moduli),tau,jnp.conj(tau),fluxes,mode="real")
#dV = model.dV(moduli,jnp.conj(moduli),tau,jnp.conj(tau),fluxes,mode="real")

ddV = model.ddV_x(x,fluxes)
dV = model.dV_x(x,fluxes)

print("Start minimisation...")
counter = 0
tic = time.time()
res = 10.
while res>tol:

    #ddV = model.ddV(moduli,jnp.conj(moduli),tau,jnp.conj(tau),fluxes,mode="real")
    #dV = model.dW(moduli,jnp.conj(moduli),tau,jnp.conj(tau),fluxes,mode="real")
    ddV = model.ddV_x(x,fluxes)
    dV = model.dV_x(x,fluxes)
    deltaPhi = -jnp.linalg.inv(ddV)@dV
    
    #moduli = moduli + step_size_Newton*(deltaPhi[h12+1:2*h12+1]+1j*deltaPhi[0:h12])
    #tau = tau + step_size_Newton*(deltaPhi[2*h12+1]+1j*deltaPhi[h12])
    x = x+step_size_Newton*deltaPhi
    
    res = jnp.sum(jnp.abs(dV))
    counter+=1
    
    print(f"residual: {res}                     counter: {counter}           ", end="\r")
    
    if counter>10**3:
        print("")
        print(f"STOPPED RUN: residual: {res}                     counter: {counter}           ")
        break
        
moduli,_,tau,_ = model._convert_real_to_complex(x)
toc = time.time()
print("")
print(f"Finished in {np.around((toc-tic)*1000,2)}ms.")
print(f"Difference to z0: {jnp.linalg.norm(z0-moduli)}")
print(f"Difference to tau0: {jnp.abs(tau0-tau)}")

Jit compiling...
Start minimisation...
residual: 9.3371122465713e-13                     counter: 355              
Finished in 196.5ms.
Difference to z0: 0.004519683439225652
Difference to tau0: 0.004048066488211778


In [25]:
model.dV(moduli,jnp.conj(moduli),tau,jnp.conj(tau),fluxes)

Array([1.10522702e-13-2.65079625e-13j, 2.07108636e-14+2.70790335e-15j,
       8.12089111e-15+8.80198692e-15j], dtype=complex128)

In [26]:
np.abs(model.V(moduli,jnp.conj(moduli),tau,jnp.conj(tau),fluxes))

np.float64(0.0042447142959288)

In [27]:
model.DW(moduli,jnp.conj(moduli),tau,jnp.conj(tau),fluxes)

Array([ 0.05563645-0.89383052j, -0.01290909-0.04216239j,
        0.1393165 -0.0204823j ], dtype=complex128)

After `jit` compilation, the minimsation is very fast (of order a few milliseconds).

We can also employ more sophisticated minimsation algorithms e.g. from the optimisation library `scipy.optimize.`.
Here, we can use [`scipy.optimize.root`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.root.html) as follows

In [28]:
from scipy.optimize import root

eps=1e-10
x0=np.array([0,1,0,1,0,1.])*2

result = root(model.dV_x,x0,args=(fluxes,), jac=model.ddV_x, method='hybr',tol=eps)

result

 message: The solution converged.
 success: True
  status: 1
     fun: [ 2.515e-14 -5.454e-15 -4.760e-15 -2.610e-15  3.803e-15
            5.525e-15]
       x: [ 1.191e+00  1.014e+00  2.938e+00  3.859e+00 -4.036e-01
            5.302e+00]
  method: hybr
    nfev: 179
    njev: 4
    fjac: [[-9.301e-01  1.385e-02 ... -5.846e-02 -2.041e-01]
           [-8.149e-02 -9.787e-01 ... -5.227e-02  8.589e-02]
           ...
           [ 5.294e-02  5.044e-02 ... -9.807e-01  1.029e-01]
           [ 2.189e-01 -1.034e-01 ... -5.262e-02 -9.340e-01]]
       r: [-9.535e+00  1.411e+00 ...  4.466e-02 -4.973e-04]
     qtf: [ 3.596e-12 -3.057e-13  2.122e-13  3.831e-13  1.348e-14
           -5.775e-14]

Let us check how close the solution is to the expected one, we compute

In [29]:
X = result.x
X = X[::2]+1j*X[1::2]

print(f"Difference to z0: {jnp.linalg.norm(z0-X[:2])}")
print(f"Difference to tau0: {jnp.abs(tau0-X[2])}")

Difference to z0: 0.004519683438488795
Difference to tau0: 0.004048066497513036


We can now verify explicitly that the above solution corresponds to a minimum of the potential by looking at the Hessian:

In [30]:
z,cz,t,ct = model._convert_real_to_complex(result.x)
H = model.hessian(z,cz,t,ct,fluxes)
H_evals = jnp.linalg.eigvalsh(H)
H_evals

Array([3.52568997e-04, 1.32232491e-01, 2.39368511e-01, 4.06306614e-01,
       4.33460483e+00, 1.00341765e+01], dtype=float64)

We can compue the masses of the fields as follows

In [31]:
M = model.mass_matrix(z,cz,t,ct,fluxes)
masses = jnp.linalg.eigvalsh(M)
masses

Array([1.60927285e-02, 5.20744503e+00, 1.96227826e+01, 2.58908334e+01,
       4.53143616e+01, 1.40395052e+02], dtype=float64)

## Projection on ISD and AISD fluxes

The three-form flux $G_3$ is naturally decomposed according to its Hodge type. It is well known that supersymmetric fluxes are of type $(2,1)$ and primitive, see [1707.01095](https://arxiv.org/pdf/1707.01095) for references. In the low-energy effective theory, this condition is reflected by the fact that, at the vacuum point in complex structure moduli space, the flux vector $\vec{N}=\vec{f}-\tau\vec{h}$ has non-vanishing components only along the $D_i \Pi$ directions.
Supersymmetry is already broken at tree level by the Kähler moduli directions by the presence of a non-vanishing $(0,3)$ piece (corresponding to $W_0 \neq 0$). The $(2,1)$-components together with the $(0,3)$-component furnish the ISD part of the complex 3-form $G_3$.

In contrast, the non-supersymmetric solutions with $D_IW\neq 0$ include, in addition to the $(2,1)$-components, components along the $(1,2)$ and $(3,0)$ directions. They form the AISD part of the complex 3-form $G_3$. 


To project the integer fluxes in our symplectic basis onto the ISD and AISD components, recall that the set $\{ \Omega, \bar{\Omega}, D_i \Omega, \bar{D}_{\bar{\imath}} \bar{\Omega} \}$ forms a basis of three-forms for generic values of the moduli, see [Candelas, de la Ossa 1990](https://inspirehep.net/literature/295485). The corresponding set $\{ \Pi, \Pi^*, D_i \Pi, \bar{D}_{\bar{\imath}} \Pi^* \}$ constitutes an orthogonal basis with respect to the symplectic inner product.As a result, any vector $\vec{V}$ in the period/charge vector space can be expanded as (see [1707.01095](https://arxiv.org/pdf/1707.01095)):
$$
\vec{V} = \mathrm{i} e^{K_{\text{cs}}} \Bigl [(\Pi^\dagger \Sigma \vec{V}) \Pi- (\Pi^T \Sigma \vec{V}) \Pi^*- (K^{\bar{\jmath} l} \bar{D}_{\bar{\jmath}} \Pi^\dagger \Sigma \vec{V}) D_l \Pi - (K^{j \bar{l}} D_j \Pi^T \Sigma \vec{V}) \bar{D}_{\bar{l}} \Pi^* \Bigl ]
$$
where $\Pi$ is evaluated at a generic point in complex structure moduli space. For the flux vector $\vec{N}$, the expansion coefficients are given by
$$
\vec{N} =  \mathrm{i} e^{K_{\text{cs}}} \left(\frac{D_{\bar{\tau}}\overline{W}}{K_{\bar{\tau}}} \Pi- W\Pi^*- \frac{(D_{\bar{\jmath}}D_{\bar{\tau}}\overline{W})}{K_{\bar{\tau}}}\, K^{i\bar{\jmath}} \, D_i \Pi- (D_i W)  K^{i\bar{\jmath}} \bar{D}_{\bar{\jmath}} \Pi^*\right)
$$


Let us compute the corresponding projections for the supersymmetric minimum from above:

In [32]:
# Moduli values
z_SUSY = jnp.array([0.3168651582886485 +3.2301486028589927j,0.37113099110064307+3.0746240438915935j])

# Axio-dilaton values
tau_SUSY = -0.29834960566944685+6.803576151333438j

# Choices of fluxes
fluxes_SUSY = jnp.array([4.,  -3.,  -2.,  -2.,   3.,   2.,  39., -13.,  -4.,   0.,  -0.,  0.])

In [33]:
N_3_0,N_2_1,N_1_2,N_0_3 = model.projection_fluxes(z_SUSY,tau_SUSY,fluxes_SUSY,mode=None)

We can check that the $(3,0)$- and $(1,2)$-components vanish

In [34]:
np.sum(np.abs(N_3_0)),np.sum(np.abs(N_1_2))

(np.float64(4.0067287943056404e-11), np.float64(2.0320398327314313e-11))

The remaining components should therefor add up to the flux vectors `fluxes_SUSY` from above

In [35]:
np.around(N_2_1+N_0_3,10)

Array([  4.,  -3.,  -2.,  -2.,   3.,   2.,  39., -13.,  -4.,  -0.,   0.,
         0.], dtype=float64)

or more explicitly

In [36]:
np.sum(np.abs(N_2_1+N_0_3-fluxes_SUSY))

np.float64(5.677622261224258e-11)

Next, let us consider the non-SUSY minimum from above

In [37]:
# Moduli values
z_nonSUSY = jnp.array([1.190671013458699 +1.0138094643117892j,2.9377413761449915+3.8593967759445964j])

# Axio-dilaton values
tau_nonSUSY = -0.403614509914007+5.301822679470445j

# Choices of fluxes
fluxes_nonSUSY = jnp.array([0., 18., 3., 2., 4., -4., 0., -7., 0., 0., 0., 1.])

We compute the projections as above

In [38]:
N_3_0,N_2_1,N_1_2,N_0_3 = model.projection_fluxes(z_nonSUSY,tau_nonSUSY,fluxes_nonSUSY,mode=None)

This time, the $(3,0)$- and $(1,2)$-components are non-vanishing

In [39]:
np.sum(np.abs(N_3_0)),np.sum(np.abs(N_1_2)),np.sum(np.abs(N_2_1)),np.sum(np.abs(N_0_3))

(np.float64(0.7829879967905087),
 np.float64(1.5087264770524165),
 np.float64(59.76871458278726),
 np.float64(78.34802488882704))

We note however that the $(3,0)$- and $(1,2)$-components are comparatively small to the ISD components.
This is because the SUSY breaking scale is rather small (see [2308.15525](https://arxiv.org/abs/2308.15525) for a discussion)
as can be seen from the value of the scalar potential at the minimum:

In [40]:
model.V(z_nonSUSY,jnp.conj(z_nonSUSY),tau_nonSUSY,jnp.conj(tau_nonSUSY),fluxes_nonSUSY)

Array(0.00424471+2.59535884e-19j, dtype=complex128)

We can again make the consistency check that the components add up to the integer quantised flux from above

In [41]:
np.around(N_3_0+N_2_1+N_1_2+N_0_3,10)

Array([ 0., 18.,  3.,  2.,  4., -4.,  0., -7., -0.,  0.,  0.,  1.],      dtype=float64)

or more explicitly

In [42]:
np.sum(np.abs(N_3_0+N_2_1+N_1_2+N_0_3-fluxes_nonSUSY))

np.float64(5.1958437552457326e-14)